[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kunifujiwara/VoxCity/blob/main/docs/examples/demo_basic.ipynb) · part of the [VoxCity tutorials](index.md)

No account is needed: the default data sources are read from open repositories. Google Earth Engine is optional and only adds further sources ([guide](../guides/earth_engine.md)).

# VoxCity Complete Demo

The full workflow with every main option: define an area, choose data sources, generate the model, visualize it, run simulations, and export.

Run the steps in order. In Step 2 and Step 3, run **one** option.

# Step 1: Prepare environment

In [ ]:
#@title 1.1. Install voxcity
%pip install -q voxcity
# Static (PyVista) rendering on Colab needs an X server and OpenGL libraries
!apt-get -qq install -y xvfb libgl1 > /dev/null 2>&1 || true

In [ ]:
#@title (Optional) 1.2. Authenticate Google Earth Engine - only for Earth Engine sources
# Google Colab: click the link, generate a token, paste it
# !earthengine authenticate --auth_mode=notebook

# Local environment
# import ee
# ee.Authenticate()
# ee.Initialize(project='your-project-id')

# Step 2: Define the target area

| Option | Method |
|--------|--------|
| 2.1 | Type the corner coordinates |
| 2.2 | Draw on a map: aligned or rotated box, or a fixed-size box around a clicked centre |

In [ ]:
#@title 2.1. Option 1: Set coordinates of vertices
rectangle_vertices = [
    (-74.02034270713835, 40.69992881162822),  # Southwest corner (longitude, latitude)
    (-74.02034270713835, 40.7111851828668),   # Northwest corner
    (-74.00555129286164, 40.7111851828668),   # Northeast corner
    (-74.00555129286164, 40.69992881162822),  # Southeast corner
]

In [ ]:
#@title 2.2. Option 2: Draw a rectangle on a map
#@markdown Use the panel: type a city, then draw (Aligned / Rotated) or click **Fixed** and click the centre.
cityname = "new york" #@param {type:"string"}
width = 1250 #@param {type:"number"}
height = 1250 #@param {type:"number"}
from voxcity.geoprocessor.draw import create_rectangle_map

m, rectangle_vertices = create_rectangle_map(cityname, zoom=15, width=width, height=height)
m

# Step 3: Generate the 3D voxel model

Option 1 selects sources by location. Option 2 lets you choose them; sources marked *EE* need Earth Engine (Step 1.2).

| Layer | No account | Earth Engine (*EE*) |
|-------|------------|---------------------|
| Buildings | OpenStreetMap, Overture, Microsoft Building Footprints, Global Building Atlas, EUBUCCO v0.1, Local file | Open Building 2.5D Temporal, England / Netherlands DSM - DTM |
| Land cover | OpenStreetMap, ESA WorldCover, OpenEarthMapJapan | Urbanwatch, ESRI 10m Annual Land Cover, Dynamic World V1 |
| Canopy height | High Resolution 1m Global Canopy Height Maps v2, OpenStreetMap, Static | High Resolution 1m Global Canopy Height Maps, ETH Global Sentinel-2 10m |
| Terrain | FABDEM, COPERNICUS, USGS 3DEP 10m, GSI DEM Japan, Flat | USGS 3DEP 1m, England 1m DTM, DEM France 1m / 5m, Netherlands 0.5m DTM, AUSTRALIA 5M DEM, DeltaDTM, NASA |

In [ ]:
#@title 3.1. Option 1: Automatically selected data sources
from voxcity.generator import get_voxcity

meshsize = 5 #@param {type:"number"}
city = get_voxcity(rectangle_vertices, meshsize, output_dir="output/basic")
print(city.extras["selected_sources"])

In [ ]:
#@title 3.2.1. Option 2: Set data sources
building_source = 'OpenStreetMap' #@param ['OpenStreetMap', 'Overture', 'Microsoft Building Footprints', 'Global Building Atlas', 'EUBUCCO v0.1', 'Open Building 2.5D Temporal', 'Local file']
building_complementary_source = 'Microsoft Building Footprints' #@param ['None', 'Microsoft Building Footprints', 'Overture', 'OpenStreetMap', 'Global Building Atlas', 'EUBUCCO v0.1', 'Open Building 2.5D Temporal', 'England 1m DSM - DTM', 'Netherlands 0.5m DSM - DTM', 'Local file']
land_cover_source = 'OpenStreetMap' #@param ['OpenStreetMap', 'ESA WorldCover', 'OpenEarthMapJapan', 'Urbanwatch', 'ESRI 10m Annual Land Cover', 'Dynamic World V1']
canopy_height_source = 'High Resolution 1m Global Canopy Height Maps v2' #@param ['High Resolution 1m Global Canopy Height Maps v2', 'High Resolution 1m Global Canopy Height Maps', 'ETH Global Sentinel-2 10m Canopy Height (2020)', 'OpenStreetMap', 'Static']
dem_source = 'USGS 3DEP 10m' #@param ['USGS 3DEP 10m', 'FABDEM', 'COPERNICUS', 'GSI DEM Japan', 'Flat', 'USGS 3DEP 1m', 'England 1m DTM', 'DEM France 1m', 'DEM France 5m', 'Netherlands 0.5m DTM', 'AUSTRALIA 5M DEM', 'DeltaDTM', 'NASA']
meshsize = 5 #@param {type:"number"}

In [ ]:
#@title 3.2.2. Optional parameters
kwargs = {
    "output_dir": "output/basic",           # where downloads and voxcity.h5 are written
    "building_complement_height": 10,       # height (m) for buildings with no height in any source
    "dem_interpolation": True,              # smooth the DEM when meshsize is finer than the DEM
    # "building_path": "buildings.gpkg",    # with building_source='Local file'
    # "complement_polygon": True,           # also add footprints found only in the complementary source
    # "remove_perimeter_object": 0.1,       # clear objects within 10% of the edges (useful for CFD)
    # "min_canopy_height": 2.0,             # ignore trees lower than this (m)
    # "trunk_height_ratio": 0.59,           # trunk height / tree height
    # "gridvis": True,                      # plot the 2D input grids
}

In [ ]:
#@title 3.2.3. Generate the model with the chosen sources
from voxcity.generator import get_voxcity

city = get_voxcity(
    rectangle_vertices,
    meshsize,
    building_source=building_source,
    building_complementary_source=building_complementary_source,
    land_cover_source=land_cover_source,
    canopy_height_source=canopy_height_source,
    dem_source=dem_source,
    **kwargs,
)

In [ ]:
#@title 3.3. What is inside the model
voxcity_grid = city.voxels.classes           # 3D grid: (north, east, up)
building_height_grid = city.buildings.heights
land_cover_grid = city.land_cover.classes
dem_grid = city.dem.elevation
canopy_height_grid = city.tree_canopy.top
building_gdf = city.extras.get("building_gdf")
print(voxcity_grid.shape, len(building_gdf) if building_gdf is not None else 0, "buildings")

In [ ]:
#@title (Optional) Load a saved model instead of generating it
from voxcity.generator import load_voxcity
city = load_voxcity("output/basic/voxcity.h5")

# Step 4: Visualize

In [ ]:
#@title 4.1. Interactive 3D view
from voxcity.visualizer import visualize_voxcity
visualize_voxcity(city, mode="interactive")

In [ ]:
#@title 4.2. Static images (slow for large areas)
from IPython.display import Image, display
from voxcity.visualizer import visualize_voxcity

images = visualize_voxcity(city, mode="static", output_directory="output/basic")
for name, path in images[:2]:
    display(Image(filename=path))

# Step 5: Simulations

In [ ]:
#@title 5.1. Ground-level solar irradiance (instantaneous)
from voxcity.simulator.solar import get_global_solar_irradiance_using_epw

solar_grid = get_global_solar_irradiance_using_epw(
    city,
    calc_type="instantaneous",
    calc_time="06-21 15:00:00",       # "MM-DD HH:MM:SS" in a typical-year weather file
    download_nearest_epw=True,        # nearest EPW from Climate.OneBuilding.Org
    output_dir="output/basic",
    view_point_height=1.5,
    colormap="magma",
)

In [ ]:
#@title 5.2. Ground-level solar irradiance over a typical year (fast sky-patch method)
cum_solar_grid = get_global_solar_irradiance_using_epw(
    city,
    calc_type="cumulative",
    start_time="01-01 01:00:00",
    end_time="12-31 23:00:00",
    use_sky_patches=True,             # groups sun positions into sky patches: seconds, not minutes
    download_nearest_epw=True,
    output_dir="output/basic",
    view_point_height=1.5,
    colormap="magma",
)

In [ ]:
#@title 5.3. Solar irradiance on building surfaces (instantaneous)
from voxcity.simulator.solar import get_building_global_solar_irradiance_using_epw
from voxcity.visualizer import visualize_voxcity

building_irradiance = get_building_global_solar_irradiance_using_epw(
    city,
    calc_type="instantaneous",
    calc_time="06-21 15:00:00",
    download_nearest_epw=True,
    output_dir="output/basic",
)
visualize_voxcity(city, mode="interactive", building_sim_mesh=building_irradiance,
                  building_value_name="global", building_colormap="magma", voxel_color_map="grayscale")

In [ ]:
#@title 5.4. Green view index and sky view index
from voxcity.simulator.view import get_view_index

gvi_grid = get_view_index(city, mode="green", view_point_height=1.5, colormap="viridis")
svi_grid = get_view_index(city, mode="sky", view_point_height=1.5, colormap="BuPu_r", elevation_min_degrees=0)

In [ ]:
#@title 5.5.1. Landmark visibility: draw a polygon around the landmark buildings
from voxcity.geoprocessor.draw import display_buildings_and_draw_polygon

m_landmark, landmark_polygon = display_buildings_and_draw_polygon(city)
m_landmark

In [ ]:
#@title 5.5.2. Landmark visibility map (buildings at the centre are used if no polygon was drawn)
from voxcity.simulator.view import get_landmark_visibility_map

landmark_vis_map, _ = get_landmark_visibility_map(
    city,
    landmark_polygon=landmark_polygon or None,
    view_point_height=1.5,
    colormap="cool",
)

In [ ]:
#@title 5.6. Aggregate a result along the street network
from voxcity.geoprocessor.network import get_network_values

G, edge_gdf = get_network_values(
    cum_solar_grid,
    voxcity=city,
    value_name="Yearly global irradiance (Wh/m²)",
    network_type="walk",
    colormap="magma",
    vis_graph=True,
)

In [ ]:
#@title 5.7. Show a ground result on the 3D model
visualize_voxcity(
    city,
    mode="interactive",
    ground_sim_grid=cum_solar_grid,
    ground_dem_grid=dem_grid,
    ground_colormap="magma",
    ground_view_point_height=1.5,
    voxel_color_map="grayscale",
)

# Step 6: Export

In [ ]:
#@title 6.1. ENVI-met (INX + plant database)
from voxcity.exporter.envimet import export_inx

export_inx(
    city,
    output_directory="output/basic/envimet",
    file_basename="voxcity",
    author_name="your name",
    model_description="generated with VoxCity",
    useTelescoping_grid=True,
    verticalStretch=20,
    min_grids_Z=20,
)   # also writes projectdatabase.edb next to the INX

In [ ]:
#@title 6.2. PALM static driver
from voxcity.exporter.palm import export_palm
export_palm(city, output_directory="output/basic/palm", domain_name="voxcity")

In [ ]:
#@title 6.3. OBJ (Blender, Rhino, Twinmotion) and MagicaVoxel VOX
from voxcity.exporter.obj import export_obj
from voxcity.exporter.magicavoxel import export_magicavoxel_vox

export_obj(city, "output/basic/obj", "voxcity")
export_magicavoxel_vox(city, "output/basic/magicavoxel")

In [ ]:
#@title 6.4. GeoTIFF layers and saving the model
from voxcity.exporter.geotiff import export_geotiffs
from voxcity.generator import save_voxcity

export_geotiffs(city, "output/basic/geotiff")
save_voxcity("output/basic/voxcity_saved.h5", city)